In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession
    .builder
    .master("local[*]")
    .appName("Spark Streaming")
    .getOrCreate()
)



In [2]:
spark

In [3]:
df_raw = spark.readStream.format("socket").option("host", "172.23.0.1").option("port", "9999").load()


In [4]:
from pyspark.sql.functions import split

df_words = df_raw.withColumn("words", split("value", " "))

In [5]:
from pyspark.sql.functions import explode

df_explode = df_words.withColumn("word", explode("words")).drop("value", "words")

In [6]:
from pyspark.sql.functions import count, lit
df_agg = df_explode.groupBy("word").agg(count(lit(1)).alias("cnt"))

In [ ]:
df_agg.writeStream.format("console").outputMode("complete").start().awaitTermination()